# glaive 파인튜닝 — 이 PC(CPU)에서 돌린 진행 상황

`glaive_colab.md`와 같은 학습을 **GPU 없이 이 PC의 CPU로** 돌리고 있습니다.
Colab(T4)에서는 클릭이 필요해서, 사람이 손대지 않고 끝까지 가는 방법으로 병행했습니다.

| | 문서 (Colab / GTX 1650) | 이 PC |
|---|---|---|
| 장치 | GPU | CPU (i7-1165G7, 4코어) |
| 모델 적재 | 4비트 양자화 (QLoRA) | fp32 그대로 (bitsandbytes는 GPU 전용) |
| 옵티마이저 | paged_adamw_8bit, fp16 | adamw_torch, fp32 |
| 모델 · 데이터 · LoRA 설정 | Qwen2.5-1.5B, 500/100건, r=8, q·k·v·o | **같음** |
| 학습 가능 파라미터 | 약 218만 (0.14%) | 2,179,072 (0.141%) |

실행 스크립트: `glaive_cpu_run.py` · 로그: `glaive_cpu_run.log`

아래 셀을 실행하면 최신 로그를 읽어 진행 상황을 다시 계산합니다.

In [1]:
import json
import os
import re
import urllib.request

LOG_URL = "https://raw.githubusercontent.com/kimdongjun22/aiwins/main/fine_tuning/glaive_cpu_run.log"

if os.path.exists("glaive_cpu_run.log"):
    log = open("glaive_cpu_run.log", encoding="utf-8").read()
    source = "로컬 파일"
else:
    log = urllib.request.urlopen(LOG_URL).read().decode("utf-8")
    source = "GitHub"

steps = re.findall(r"\[step (\d+)/(\d+)\] 스텝당 (\d+)초, 남은 시간 약 (\d+)분", log)
done, total, per, left = map(int, steps[-1])
print(f"[{source}] 진행 {done}/{total} 스텝 ({done / total:.0%})")
print(f"스텝당 평균 {per}초, 남은 시간 약 {left // 60}시간 {left % 60}분")
print("학습 끝" if "학습 끝" in log else "학습 진행 중")

[로컬 파일] 진행 20/63 스텝 (32%)
스텝당 평균 245초, 남은 시간 약 2시간 56분
학습 진행 중


## 학습 손실

문서 값은 GTX 1650에서 잰 것입니다. 10스텝마다 한 번 찍힙니다.

In [2]:
DOC = {10: 0.64, 30: 0.56, 60: 0.58}

rows = []
for m in re.finditer(r"\[log\] (\{.*?\})\n", log):
    rec = json.loads(m.group(1))
    if "loss" in rec:
        step = round(rec["epoch"] * 63)
        rows.append((step, rec["loss"], rec.get("mean_token_accuracy")))

print(f"{'스텝':>4} | {'이 PC 손실':>9} | {'문서 손실':>8} | {'토큰 정확도':>9}")
print("-" * 44)
for step, loss, acc in rows:
    doc = DOC.get(step)
    print(f"{step:>4} | {loss:>9.3f} | {doc if doc else '-':>8} | {acc:>9.1%}")

ev = re.search(r'"eval_loss": ([0-9.]+)', log)
print("\n검증 손실:", ev.group(1) if ev else "아직 (학습이 끝나면 계산)", "/ 문서 0.56")

  스텝 |   이 PC 손실 |    문서 손실 |    토큰 정확도
--------------------------------------------
  10 |     0.620 |     0.64 |     80.9%
  20 |     0.520 |        - |     82.7%

검증 손실: 아직 (학습이 끝나면 계산) / 문서 0.56


## 질의 결과 (23~26단계)

학습이 끝나면 같은 질문에 대한 세 가지 답이 여기 나옵니다: 파인튜닝한 모델, 데이터셋의 답, 원래 Qwen.

In [3]:
part = log.split("[23단계]", 1)
print("[23단계]" + part[1] if len(part) == 2 else "아직 학습 중입니다. 끝나면 이 셀을 다시 실행하세요.")

아직 학습 중입니다. 끝나면 이 셀을 다시 실행하세요.
